# Stage 7 — Get the encoder running

**The question:** can we turn tiles into feature vectors, both online and
offline?

Each tile goes through UNI2-h and comes out as 1536 numbers. Getting that to
work is easy. The hard part is the word **offline**: a Kaggle submission runs
with the internet switched off, so a model that can only be fetched from
HuggingFace cannot be used to submit. This notebook proves the offline path
works now, not on submission day.

What it does:

1. Loads UNI2-h from HuggingFace (the online path) and saves its weights to a
   local file, once.
2. Defines the tile-to-numbers transform in exactly one place, inside
   `encode_tiles`.
3. Loads the same weights from the local file with the network blocked (the
   offline path), and checks both paths give identical features.
4. Times tiling plus encoding per slide, and scales it up to the hidden test
   set.

The encoder code lives in `src/encode.py` and the tile selector in
`src/tiling.py`; this notebook runs and checks them. It writes one file,
`data/weights/uni2h.pt`, and only if it does not exist yet.
That file is git-ignored. UNI2-h's licence (CC-BY-NC-ND) forbids sharing it, so
it only ever goes to Kaggle as a **private** dataset.

**On Kaggle** the online path is skipped: there is no internet at submission
time, and that is the situation being tested. The weights come from the
attached private dataset instead.

In [1]:
"""Imports.

The first block makes the project's src/ folder importable. Locally it sits
one folder up from notebooks/. On Kaggle it arrives as an attached dataset, so
every attached dataset is checked for a src/ folder, and the notebook stops
with a clear message if there is none.

contextlib makes the "no network" block used to test the offline path.
gc         frees the online model's memory before the offline one is loaded.
socket     is what the "no network" block switches off.
time       times tiling and encoding.
pandas     loads the inventory and holds the timing results.
torch      checks the features.
tqdm       draws a progress bar for the timing loop.
src.paths   where the data and the weights are, locally and on Kaggle.
src.tiling  get_tiles, the tile selector.
src.encode  the two loading paths, prepare and encode_tiles.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

import contextlib
import gc
import socket
import time

import pandas as pd
import torch
from tqdm import tqdm

from src.encode import (
    BATCH_SIZE,
    DEVICE,
    FEATURE_DIM,
    MEAN,
    STD,
    encode_tiles,
    load_encoder_offline,
    load_encoder_online,
    prepare,
)
from src.paths import ON_KAGGLE, find_input, slide_path, weights_path
from src.tiling import TILE_SIZE, get_tiles


/home/aztrek/projects/Prostate-Cancer-Grading/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""Inputs and settings.

1. INVENTORY_PATH is the Stage 1 inventory, used only to pick slides.
2. WEIGHTS_PATH is the local copy of the UNI2-h weights. Locally this notebook
   writes it; on Kaggle it must already be attached as a private dataset.
3. SEED fixes which slides are picked.
4. TIMING_SAMPLE slides per hospital are timed.
5. TEST_SLIDES is the rough size of the hidden test set, from the handbook.
6. TIME_LIMIT_HOURS is the submission notebook's time limit. 9 hours is the
   usual limit for Kaggle code competitions, but we have not confirmed it for
   PANDA: check the competition's Code Requirements page.

The encoder's own settings (architecture, normalisation, precision, batch
size) live in src/encode.py.
"""

INVENTORY_PATH = find_input("slide_inventory.parquet")
WEIGHTS_PATH = weights_path()

SEED = 0
TIMING_SAMPLE = 10
TEST_SLIDES = 1000
TIME_LIMIT_HOURS = 9

print(f"device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

slide_inventory.parquet: /home/aztrek/projects/Prostate-Cancer-Grading/data/derived/slide_inventory.parquet
device: cuda (NVIDIA GeForce RTX 4060)


## 1. Two ways to load UNI2-h

**Online:** ask HuggingFace for the model by name. timm downloads the weights
(or reuses its local cache) and needs a HuggingFace login, because UNI2-h is
gated. This is how the weights get onto our machine the first time.

**Offline:** build the empty network from its description in
`src/encode.py`, then fill it from the local weights file. No step asks the internet for
anything. This is the path every Kaggle notebook uses.

**Precision.** On the GPU the model runs in fp16 (half precision): it halves
the memory, from 2.7 GB to 1.4 GB, which is what lets it fit a 4 GB laptop
card, and it is faster. The same setting is used everywhere, locally and on
Kaggle, so training features and submission features come from identical
arithmetic. A check below measures how much fp16 changes the features compared
with full precision.

Both are in `src/encode.py`, as `load_encoder_online`, `load_encoder_offline`
and `prepare`.

## 2. The transform, in exactly one place

Before the model sees a tile, its pixels are scaled from 0-255 to 0-1 and
normalised with the model's MEAN and STD. That step lives **inside**
`encode_tiles` in `src/encode.py`, and nowhere else. If it were written once for feature
extraction and again for the submission notebook, the two would drift apart
one day (a different mean, a different resize) and the model would get worse
with no error message.

There is no resize: tiles are already 224 x 224. A tile of any other size is a
bug upstream, so `encode_tiles` stops with an error instead of quietly
resizing it.

The output is converted to fp16 explicitly. The model's last layer is a
LayerNorm, which PyTorch may compute in fp32, so the output type is not assumed.

## 3. The online path

Skipped on Kaggle. Locally it needs a HuggingFace login once, in a terminal:
`.venv/bin/python -m huggingface_hub.commands.huggingface_cli login`. Never
paste the token into a notebook.

In [3]:
"""Load UNI2-h from HuggingFace, check its settings, and save the weights file.

1. Load the model online, in fp32 on the CPU.
2. Check the model's own settings match ours: the input size, and the MEAN and
   STD that encode_tiles normalises with. A mismatch here would quietly spoil
   every feature.
3. Count its parameters: UNI2-h has about 681 million.
4. Save its weights to WEIGHTS_PATH, only if the file is not there yet. The
   save goes to a temporary name first and is renamed at the end, so a save
   that is interrupted never leaves a broken file that looks finished.
"""

if ON_KAGGLE:
    print("on Kaggle: online path skipped; weights come from the attached dataset")
else:
    online = load_encoder_online()

    config = online.pretrained_cfg
    assert tuple(config["input_size"]) == (3, TILE_SIZE, TILE_SIZE), config["input_size"]
    assert tuple(config["mean"]) == MEAN and tuple(config["std"]) == STD, (config["mean"], config["std"])
    print(f"model settings match: input {config['input_size']}, mean {MEAN}, std {STD}")
    print(f"parameters: {sum(p.numel() for p in online.parameters()) / 1e6:.0f} M")

    if os.path.exists(WEIGHTS_PATH):
        print(f"{WEIGHTS_PATH} already exists ({os.path.getsize(WEIGHTS_PATH) / 1e9:.2f} GB), not rewritten")
    else:
        os.makedirs(os.path.dirname(WEIGHTS_PATH), exist_ok=True)
        torch.save(online.state_dict(), WEIGHTS_PATH + ".part")
        os.replace(WEIGHTS_PATH + ".part", WEIGHTS_PATH)
        print(f"wrote {WEIGHTS_PATH} ({os.path.getsize(WEIGHTS_PATH) / 1e9:.2f} GB)")

model settings match: input [3, 224, 224], mean (0.485, 0.456, 0.406), std (0.229, 0.224, 0.225)
parameters: 681 M
wrote /home/aztrek/projects/Prostate-Cancer-Grading/data/weights/uni2h.pt (2.73 GB)


In [4]:
"""Run real tiles through the online model and check what comes out.

1. Take 16 tiles from one random slide per hospital: 32 tiles to test with.
2. Encode 4 of them at full precision on the CPU. This is the reference for
   the fp16 check.
3. Make the model ready (fp16 on the GPU) and encode all 32.
4. Check the output: shape [32, 1536], dtype float16, no infinities or NaNs.
   Print the value range and the typical length of a feature vector. A model
   with random weights gives a much narrower range; if a later run looks like
   that, the weights did not load.
5. Compare fp16 against full precision on the 4 reference tiles, by cosine
   similarity: 1.0 means the same direction. Values very close to 1 mean fp16
   costs nothing that matters.
"""

inventory = pd.read_parquet(INVENTORY_PATH)
check_slides = [
    inventory[inventory["provider"] == provider].sample(1, random_state=SEED)["slide_id"].iloc[0]
    for provider in ("radboud", "karolinska")
]
check_tiles = []
for slide_id in check_slides:
    check_tiles += get_tiles(slide_path(slide_id))[0][:16]
print(f"{len(check_tiles)} test tiles from slides {[s[:8] for s in check_slides]}")

if not ON_KAGGLE:
    reference = encode_tiles(online, check_tiles[:4])

    online = prepare(online)
    online_features = encode_tiles(online, check_tiles)

    assert online_features.shape == (len(check_tiles), FEATURE_DIM), online_features.shape
    assert online_features.dtype == torch.float16, online_features.dtype
    assert torch.isfinite(online_features).all(), "non-finite features"
    print(f"features: shape {tuple(online_features.shape)}, dtype {online_features.dtype}, all finite")
    print(f"value range: {online_features.min().item():.3f} to {online_features.max().item():.3f}")
    print(f"vector length, median: {online_features.float().norm(dim=1).median().item():.2f}")

    cosine = torch.nn.functional.cosine_similarity(reference.float(), online_features[:4].float(), dim=1)
    print(f"fp16 against full precision, cosine similarity: lowest {cosine.min().item():.6f}")

32 test tiles from slides ['5af44705', '21c61eab']
features: shape (32, 1536), dtype torch.float16, all finite
value range: -2.215 to 2.104
vector length, median: 16.25
fp16 against full precision, cosine similarity: lowest 0.999976


## 4. The offline path, with the network switched off

The online model is deleted first, so nothing can be borrowed from it. Then
the weights are loaded from the local file **inside a block that makes every
network connection fail**. If any step of loading or encoding tried to reach
the internet, it would stop with an error here rather than on submission day.
The block first checks itself by trying to reach HuggingFace, which must fail.

This blocks Python's own connections, which is how timm and HuggingFace reach
the internet. The final proof is on Kaggle: attach the weights as a private
dataset, switch the notebook's internet off, and run this notebook there.

Locally the offline features must equal the online features **exactly**, not
approximately: the same weights and the same arithmetic on the same tiles
leave no room for any difference.

In [5]:
"""Load from the local file with the network blocked, and compare.

no_network: inside the block, every attempt to open a network connection
raises an error. On leaving it, connections work again.

1. Delete the online model and free its memory.
2. Inside the block: confirm the block works, load the model from
   WEIGHTS_PATH, make it ready, and encode the same 32 tiles.
3. Encode them a second time and require identical output: the same tile must
   always give the same features.
4. Locally, require the offline features to equal the online ones exactly.
"""

@contextlib.contextmanager
def no_network():
    """Make every network connection attempt fail inside the block."""
    real_connect = socket.socket.connect

    def refuse(*args, **kwargs):
        raise OSError("network is switched off for this check")

    socket.socket.connect = refuse
    try:
        yield
    finally:
        socket.socket.connect = real_connect


if not ON_KAGGLE:
    del online
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

with no_network():
    try:
        socket.create_connection(("huggingface.co", 443), timeout=5)
        raise AssertionError("the network block did not block anything")
    except OSError:
        print("network blocked")
    encoder = prepare(load_encoder_offline(WEIGHTS_PATH))
    offline_features = encode_tiles(encoder, check_tiles)
print(f"loaded from {WEIGHTS_PATH} with no network")

assert torch.equal(offline_features, encode_tiles(encoder, check_tiles)), "two runs gave different features"
print("repeatable: two runs on the same tiles are identical")

if not ON_KAGGLE:
    difference = (offline_features.float() - online_features.float()).abs().max().item()
    print(f"largest difference between the online and offline paths: {difference}")
    assert torch.equal(offline_features, online_features), "offline path does not match online"
    print("offline path matches the online path exactly")

network blocked
loaded from /home/aztrek/projects/Prostate-Cancer-Grading/data/weights/uni2h.pt with no network
repeatable: two runs on the same tiles are identical
largest difference between the online and offline paths: 0.0
offline path matches the online path exactly


## 5. Time it

The number that matters at submission: seconds per slide for tiling plus
encoding, from the slide file to the features. Multiplied by the size of the
hidden test set, it must fit well inside the notebook's time limit. The
handbook asks for 40% headroom, so the target is 60% of the limit.

**Warm cache.** A slide read a second time comes from memory and looks faster
than it is. The timed slides are drawn with a different seed from every other
sample in this notebook, so this notebook has not read them before. Earlier
notebooks may have read parts of them, so reading time could still come out a
little optimistic.

The GPU is warmed up first on the test tiles, because its very first batch
pays one-off start-up costs that no later slide pays.

In [6]:
"""Time tiling and encoding, slide by slide.

1. Warm up: encode one batch and throw the result away.
2. Draw TIMING_SAMPLE slides per hospital, with seed SEED + 1 so none of the
   slides used above is picked by design.
3. For each slide, time get_tiles and encode_tiles separately.
   encode_tiles returns features on the CPU, which waits for the GPU to
   finish, so the time is real.
4. Print the averages per hospital, then scale the overall average up to the
   hidden test set and to the whole training set, and compare the test-set
   figure with 60% of the time limit.
"""

encode_tiles(encoder, check_tiles[:BATCH_SIZE])

timed = pd.concat(
    [
        inventory[inventory["provider"] == provider].sample(TIMING_SAMPLE, random_state=SEED + 1)
        for provider in ("radboud", "karolinska")
    ]
)

timing_rows = []
for r in tqdm(list(timed.itertuples()), desc="timing"):
    start = time.perf_counter()
    tiles, _, _ = get_tiles(slide_path(r.slide_id))
    tiled = time.perf_counter()
    features = encode_tiles(encoder, tiles)
    encoded = time.perf_counter()
    assert features.shape == (len(tiles), FEATURE_DIM)
    timing_rows.append(
        {"provider": r.provider, "tiles": len(tiles), "tiling_s": tiled - start, "encoding_s": encoded - tiled}
    )

timing = pd.DataFrame(timing_rows)
timing["total_s"] = timing["tiling_s"] + timing["encoding_s"]
print(timing.groupby("provider")[["tiles", "tiling_s", "encoding_s", "total_s"]].mean().round(2).to_string())

per_slide = timing["total_s"].mean()
test_hours = TEST_SLIDES * per_slide / 3600
print(
    f"\nmean {per_slide:.2f} s per slide "
    f"({timing['tiling_s'].mean():.2f} tiling + {timing['encoding_s'].mean():.2f} encoding)"
)
print(f"hidden test set, about {TEST_SLIDES} slides: {test_hours * 60:.0f} min")
print(f"whole training set, {len(inventory)} slides: {len(inventory) * per_slide / 3600:.1f} h")
print(
    f"60% of a {TIME_LIMIT_HOURS} h limit is {0.6 * TIME_LIMIT_HOURS:.1f} h: "
    f"{'fits' if test_hours <= 0.6 * TIME_LIMIT_HOURS else 'does NOT fit'}"
)

timing: 100%|██████████| 20/20 [00:26<00:00,  1.31s/it]

            tiles  tiling_s  encoding_s  total_s
provider                                        
karolinska   76.9      0.21        1.25     1.46
radboud      65.7      0.08        1.08     1.16

mean 1.31 s per slide (0.15 tiling + 1.16 encoding)
hidden test set, about 1000 slides: 22 min
whole training set, 10616 slides: 3.9 h
60% of a 9 h limit is 5.4 h: fits


## Result

Filled in from the run on 2026-09-30, on an RTX A1000 laptop GPU (4 GB). The
outputs saved above are from the 2026-10-14 re-run below; the laptop run's
outputs are in the git history.

- The model's own settings match ours: 224 x 224 input, ImageNet mean and
  standard deviation. 681 million parameters.
- 32 test tiles give float16 features of shape [32, 1536], all finite, ranging
  from about -2.2 to 2.1.
- fp16 against full precision: lowest cosine similarity 0.999977. Running in
  fp16 changes nothing that matters.
- The offline path loads with the network blocked and gives features
  identical to the online path (largest difference 0.0). Two runs on the same
  tiles are identical.
- Tiling plus encoding: 1.96 s per slide (0.19 s tiling, 1.77 s encoding),
  over 20 slides. Karolinska slides take longer (2.22 s, 77 tiles) than
  Radboud (1.70 s, 66 tiles).
- Scaled up on this GPU: about 33 minutes for a 1,000-slide test set, well
  inside 60% of a 9-hour limit, and about 5.8 hours to extract the whole
  training set.

Re-run 2026-10-14 on a desktop RTX 4060 (8 GB): every check passes, and
offline matches online exactly. The features are not bit-identical to the
laptop's: lowest value -2.215 against -2.213, lowest fp16 cosine 0.999976
against 0.999977, from the two GPUs' fp16 arithmetic. Tiling plus encoding
1.31 s per slide (0.15 s tiling, 1.16 s encoding) over 20 slides; Karolinska
1.46 s, Radboud 1.16 s. Scaled up: about 22 min for 1,000 test slides, 3.9 h
for the training set. Weights sha256 fabfb136…16bff.

## Before Stage 12

1. Upload `data/weights/uni2h.pt` to Kaggle as a **private** dataset. Never
   make it public: the licence forbids redistribution.
2. Attach it, the Stage 1 outputs and the project's `src/` folder to a copy of
   this notebook on Kaggle,
   switch the notebook's internet **off**, and run it. It must load the
   weights, pass the repeat check, and print a timing. That is the real
   offline test; the block in section 4 is its local stand-in.
3. Kaggle's GPU differs from this laptop's, so re-read the timing there before
   relying on it.